# SIMCA peanut detection : clean calibration / validation / test workflow

- 2 calibration batches for fitting the one-class peanut SIMCA model;
- 1 validation batch for hyperparameter orientation;
- 1 final test batch, used only after selecting the configuration;
- empirical decision thresholds fixed at the 95% empirical quantile of the cross-validated target-class distribution;
- ranking focused on minimizing false negatives first, then false positives, with accuracy and F1 score as additional comparison metrics.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database_h5 import load_nir_uco_h5
from src.pixel_projection import (
    add_pixel_truth_labels,
    object_threshold_grid,
    binary_detection_metrics,
    plot_pixel_error_overlay,
    plot_pixel_fp_fn_overlay,
)
from src.simca_cv_calibration import (
    calibrate_simca_thresholds_cv,
    fit_final_simca_model,
    project_pixels_with_rule_variants,
    summarize_cv_calibration,
    run_simca_empirical_rule_grid,
)
from src.border_decision import (
    aggregate_pixel_predictions_to_objects_core,
    border_width_object_threshold_grid,
    summarize_pixel_errors_by_border_zone,
)

pd.set_option("display.max_columns", 160)
RANDOM_STATE = 42

## 1. Load database

In [3]:
H5_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "processed"
    / "nir_uco_all_images_reflectance.h5"
)

object_db, image_db = load_nir_uco_h5(
    H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

wavelengths = np.linspace(889, 1702, 69)[6:]

print("n objects:", len(object_db))
print("n images:", len(image_db))

n objects: 1262
n images: 48


## 2. Experimental split

The split below is deliberately strict:

- calibration: peanut pure objects from batches 1 and 2;
- validation: pure almond + pure peanut objects from batch 3;
- test: pure almond + pure peanut objects from batch 4.

The validation batch is used for hyperparameter orientation. The test batch is not used until the final evaluation.

In [4]:
CALIBRATION_BATCHES = [1, 2]
VALIDATION_BATCH = 3
TEST_BATCH = 4

TRAIN_FILTERS_CALIBRATION = {
    "sample_kind": ["pure"],
    "object_nut_type": ["peanut"],
    "batch": CALIBRATION_BATCHES,
}

VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": ["almond", "peanut"],
    "batch": [VALIDATION_BATCH],
}

TEST_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": ["almond", "peanut"],
    "batch": [TEST_BATCH],
}

PROJECTION_FILTERS_MIXTURES = {
    "sample_kind": ["mixture"],
}

## 3. Metrics and hierarchical ranking

The selection objective is not “best possible result”; it is an orientation of hyperparameters. We therefore use a transparent hierarchy:

1. minimize false-negative rate (`fn_rate`);
2. then minimize false-positive rate (`fp_rate`);
3. then maximize F1 score;
4. then maximize accuracy and balanced accuracy.

For scalar scores and Optuna-like ranking, the recommended cost is:

$$
\mathrm{cost} = 10 	\times \mathrm{FN\ rate} + 1 	\times \mathrm{FP\ rate}
$$

The 10:1 ratio makes false negatives the dominant criterion, without completely ignoring false positives.

In [5]:
def add_orientation_columns(df):
    out = df.copy()

    if "fn_rate" not in out.columns and "peanut_sensitivity" in out.columns:
        out["fn_rate"] = 1.0 - out["peanut_sensitivity"]
    if "fp_rate" not in out.columns and "almond_specificity" in out.columns:
        out["fp_rate"] = 1.0 - out["almond_specificity"]

    for col in ["f1_score", "accuracy", "balanced_accuracy"]:
        if col not in out.columns:
            out[col] = np.nan

    out["fn_fp_cost_10_1"] = 10.0 * out["fn_rate"] + out["fp_rate"]
    out["orientation_score"] = -out["fn_fp_cost_10_1"] + 0.05 * out["f1_score"].fillna(0) + 0.02 * out["accuracy"].fillna(0)
    return out


def rank_simca_results(df):
    out = add_orientation_columns(df)
    sort_cols = [
        "balanced_accuracy",
        "fn_rate",
        "fp_rate",
        "f1_score",
        "accuracy",
        "orientation_score",
    ]
    return out.sort_values(
        sort_cols,
        ascending=[False, True, True, False, False, False],
    ).reset_index(drop=True)


DISPLAY_COLS = [
    "preprocessing", "n_components", "rule_variant", "object_threshold",
    "tp", "fn", "fp", "tn", "fn_rate", "fp_rate",
    "peanut_sensitivity", "almond_specificity", "accuracy", "f1_score", "balanced_accuracy",
    "orientation_score", "cv_target_rejection_rate", "cv_rule_limit",
]

## 4. Hyperparameter grid on validation batch only

For empirical decision rules, `alpha=0.05` means that thresholds are the 95% quantile of the empirical cross-validated target-class statistics.

Recommended search space:

- preprocessing: keep a small, interpretable set;
- number of PCA components: local range around plausible values;
- rule variant: compare theoretical references and empirical 95% rules;
- object threshold: fixed here first, then refined separately using the validation batch.

In [ ]:
SEARCH_PREPROCESSING_CONFIGS = {
    "raw": (),
    "absorbance": ("absorbance",),
    "snv": ("snv",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
}

RULE_VARIANTS_FOR_SELECTION = [
    # theoretical references
    "simple_chi2",
    "alternative_chi2_fixed2",
    "data_driven_chi2",
    # empirical 95% CV rules
    "simple_emp_cv",
    "alternative_chi2_emp_cv",
    "data_driven_emp_cv",
]

N_COMPONENTS_GRID = [4, 6, 7, 8, 9, 10, 11]

FIXED_MATRIX_METHOD = "balanced_pixels"
FIXED_ALPHA = 0.05          # empirical quantile = 95%
FIXED_M = 40
FIXED_OBJECT_THRESHOLD = 0.75
FIXED_SG_WINDOW_LENGTH = 11
FIXED_SG_POLYORDER = 2
FIXED_POSITION_DILATION_RADIUS = 3
CV_N_SPLITS = None          # None = Leave-One-Object-Out; use 5 for a faster GroupKFold

In [7]:
validation_grid_df, validation_grid_results, validation_grid_errors = run_simca_empirical_rule_grid(
    object_db=object_db,
    image_db=image_db,
    train_filters=TRAIN_FILTERS_CALIBRATION,
    projection_filters=VALIDATION_FILTERS,
    preprocessing_configs=SEARCH_PREPROCESSING_CONFIGS,
    rule_variants=RULE_VARIANTS_FOR_SELECTION,
    n_components_values=N_COMPONENTS_GRID,
    matrix_method=FIXED_MATRIX_METHOD,
    alpha=FIXED_ALPHA,
    object_threshold=FIXED_OBJECT_THRESHOLD,
    m=FIXED_M,
    random_state=RANDOM_STATE,
    replace=False,
    wavelengths=wavelengths,
    sg_window_length=FIXED_SG_WINDOW_LENGTH,
    sg_polyorder=FIXED_SG_POLYORDER,
    position_dilation_radius=FIXED_POSITION_DILATION_RADIUS,
    cv_n_splits=CV_N_SPLITS,
    group_col="object_id",
    keep_pixel_tables=False,
    keep_cv_tables=False,
    verbose=True,
)




[1/66] preprocessing=raw | A=3 | alpha=0.05

[2/66] preprocessing=raw | A=4 | alpha=0.05

[3/66] preprocessing=raw | A=6 | alpha=0.05

[4/66] preprocessing=raw | A=7 | alpha=0.05

[5/66] preprocessing=raw | A=8 | alpha=0.05

[6/66] preprocessing=raw | A=9 | alpha=0.05

[7/66] preprocessing=raw | A=10 | alpha=0.05

[8/66] preprocessing=raw | A=11 | alpha=0.05

[9/66] preprocessing=raw | A=12 | alpha=0.05

[10/66] preprocessing=raw | A=14 | alpha=0.05

[11/66] preprocessing=raw | A=16 | alpha=0.05

[12/66] preprocessing=absorbance | A=3 | alpha=0.05

[13/66] preprocessing=absorbance | A=4 | alpha=0.05

[14/66] preprocessing=absorbance | A=6 | alpha=0.05

[15/66] preprocessing=absorbance | A=7 | alpha=0.05

[16/66] preprocessing=absorbance | A=8 | alpha=0.05

[17/66] preprocessing=absorbance | A=9 | alpha=0.05

[18/66] preprocessing=absorbance | A=10 | alpha=0.05

[19/66] preprocessing=absorbance | A=11 | alpha=0.05

[20/66] preprocessing=absorbance | A=12 | alpha=0.05

[21/66] preproces

In [8]:
validation_grid_ranked = rank_simca_results(validation_grid_df)

print("Errors:")
display(validation_grid_errors)

available_cols = [c for c in DISPLAY_COLS if c in validation_grid_ranked.columns]
display(validation_grid_ranked[available_cols].head(30))

Errors:


""


,preprocessing,n_components,rule_variant,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,orientation_score,cv_target_rejection_rate,cv_rule_limit
0,absorbance_sg_d1,7,data_driven_emp_cv,0.75,51.0,2.0,5.0,50.0,0.037736,0.090909,0.962264,0.909091,0.935185,0.935780,0.935678,-0.402775,0.050092,151.312440
1,absorbance_sg_d1,6,simple_emp_cv,0.75,53.0,0.0,10.0,45.0,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,-0.117980,0.050092,2.082295
2,absorbance_sg_d1,6,data_driven_emp_cv,0.75,53.0,0.0,10.0,45.0,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,-0.117980,0.050092,158.208911
3,absorbance_sg_d1,7,simple_emp_cv,0.75,50.0,3.0,7.0,48.0,0.056604,0.127273,0.943396,0.872727,0.907407,0.909091,0.908062,-0.629708,0.050092,1.994875
4,absorbance_snv_sg_d1,11,alternative_chi2_fixed2,0.75,49.0,4.0,6.0,49.0,0.075472,0.109091,0.924528,0.890909,0.907407,0.907407,0.907719,-0.800289,0.128770,2.000000
5,absorbance_snv_sg_d1,7,simple_chi2,0.75,46.0,7.0,4.0,51.0,0.132075,0.072727,0.867925,0.927273,0.898148,0.893204,0.897599,-1.330859,0.121427,1.000000
6,absorbance_sg_d1,7,alternative_chi2_fixed2,0.75,44.0,9.0,2.0,53.0,0.169811,0.036364,0.830189,0.963636,0.898148,0.888889,0.896913,-1.672069,0.112510,2.000000
7,absorbance_sg_d1,11,simple_emp_cv,0.75,53.0,0.0,12.0,43.0,0.000000,0.218182,1.000000,0.781818,0.888889,0.898305,0.890909,-0.155489,0.050092,1.797677
8,absorbance_sg_d1,11,data_driven_emp_cv,0.75,53.0,0.0,12.0,43.0,0.000000,0.218182,1.000000,0.781818,0.888889,0.898305,0.890909,-0.155489,0.050092,136.654622
9,absorbance_sg_d1,7,alternative_chi2_emp_cv,0.75,52.0,1.0,11.0,44.0,0.018868,0.200000,0.981132,0.800000,0.888889,0.896552,0.890566,-0.326074,0.050092,2.787501


## 5. Summaries: orientation rather than final performance

These tables answer: “which hyperparameters tend to reduce FN first, then FP?” They should not be presented as final performance estimates.

In [9]:
def summarize_orientation(df, group_cols):
    d = add_orientation_columns(df)
    summary = (
        d.groupby(group_cols, as_index=False)
        .agg(
            mean_fn_rate=("fn_rate", "mean"),
            mean_fp_rate=("fp_rate", "mean"),
            mean_f1=("f1_score", "mean"),
            mean_accuracy=("accuracy", "mean"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            mean_orientation_score=("orientation_score", "mean"),
            n_configs=("orientation_score", "size"),
        )
    )
    return summary.sort_values(
        ["mean_fn_rate", "mean_fp_rate", "mean_f1", "mean_accuracy"],
        ascending=[True, True, False, False],
    ).reset_index(drop=True)

preproc_orientation = summarize_orientation(validation_grid_ranked, ["preprocessing"])
component_orientation = summarize_orientation(validation_grid_ranked, ["n_components"])
rule_orientation = summarize_orientation(validation_grid_ranked, ["rule_variant"])

print("Preprocessing orientation")
display(preproc_orientation)
print("Number of components orientation")
display(component_orientation)
print("Decision rule orientation")
display(rule_orientation)

Preprocessing orientation


,preprocessing,mean_fn_rate,mean_fp_rate,mean_f1,mean_accuracy,mean_balanced_accuracy,mean_orientation_score,n_configs
0,snv,0.040309,0.850689,0.677550,0.546998,0.554501,-1.208959,66
1,absorbance_snv,0.050600,0.809091,0.685510,0.563131,0.570154,-1.269556,66
2,absorbance_snv_sg_d1,0.065180,0.677961,0.717321,0.622755,0.628429,-1.281441,66
3,raw,0.075758,0.546556,0.748171,0.684484,0.688843,-1.253034,66
4,absorbance,0.113493,0.509642,0.729865,0.684764,0.688432,-1.594388,66
5,absorbance_sg_d1,0.228130,0.266116,0.733109,0.752525,0.752877,-2.495713,66


Number of components orientation


,n_components,mean_fn_rate,mean_fp_rate,mean_f1,mean_accuracy,mean_balanced_accuracy,mean_orientation_score,n_configs
0,3,0.059224,0.873737,0.659382,0.525977,0.533519,-1.422492,36
1,4,0.063417,0.798485,0.679709,0.562243,0.569049,-1.387426,36
2,6,0.067610,0.603535,0.735112,0.659465,0.664427,-1.229691,36
3,7,0.075996,0.542424,0.750711,0.686471,0.690790,-1.251117,36
4,8,0.092243,0.551515,0.735963,0.673868,0.678121,-1.423672,36
5,9,0.094340,0.537374,0.737441,0.680041,0.684143,-1.430297,36
6,10,0.113732,0.514141,0.735081,0.682356,0.686063,-1.601057,36
7,16,0.117925,0.596465,0.702150,0.638374,0.642805,-1.727835,36
8,12,0.118973,0.592424,0.703119,0.639918,0.644302,-1.734197,36
9,14,0.122642,0.602020,0.697523,0.633230,0.637669,-1.780895,36


Decision rule orientation


,rule_variant,mean_fn_rate,mean_fp_rate,mean_f1,mean_accuracy,mean_balanced_accuracy,mean_orientation_score,n_configs
0,alternative_chi2_emp_cv,0.009148,0.780716,0.713288,0.597924,0.605068,-0.824574,66
1,data_driven_emp_cv,0.009148,0.782920,0.713187,0.596801,0.603966,-0.826806,66
2,simple_emp_cv,0.011435,0.775482,0.714028,0.599467,0.606541,-0.842142,66
3,alternative_chi2_fixed2,0.070040,0.592287,0.739387,0.664001,0.668837,-1.242437,66
4,data_driven_chi2,0.197827,0.417355,0.715546,0.690376,0.692409,-2.346044,66
5,simple_chi2,0.275872,0.311295,0.696090,0.706089,0.706417,-3.021088,66


In [10]:
fig = px.scatter(
    validation_grid_ranked,
    x="fp_rate",
    y="fn_rate",
    color="rule_variant",
    symbol="preprocessing",
    hover_data=["n_components", "accuracy", "f1_score", "balanced_accuracy", "tp", "fn", "fp", "tn"],
    title="Validation orientation: FN rate first, then FP rate",
)
fig.update_layout(xaxis_title="False-positive rate", yaxis_title="False-negative rate")
fig.show()

## 6. Select spectral SIMCA configuration on validation

The selected row is the first row after hierarchical ranking. You can override it manually, but do not inspect the test batch while doing so.

In [12]:
SELECTED_ROW = validation_grid_ranked.iloc[2].copy()

SELECTED_PREPROCESSING = str(SELECTED_ROW["preprocessing"])
SELECTED_PREPROCESSING_STEPS = SEARCH_PREPROCESSING_CONFIGS[SELECTED_PREPROCESSING]
SELECTED_N_COMPONENTS = int(SELECTED_ROW["n_components"])
SELECTED_RULE_VARIANT = str(SELECTED_ROW["rule_variant"])
SELECTED_ALPHA = FIXED_ALPHA

print("Selected spectral configuration from validation:")
print("preprocessing:", SELECTED_PREPROCESSING, SELECTED_PREPROCESSING_STEPS)
print("n_components:", SELECTED_N_COMPONENTS)
print("rule_variant:", SELECTED_RULE_VARIANT)
print("alpha / empirical quantile:", SELECTED_ALPHA, "/", 1 - SELECTED_ALPHA)

display(pd.DataFrame([SELECTED_ROW])[available_cols])

Selected spectral configuration from validation:
preprocessing: absorbance_sg_d1 ('absorbance', 'sg_d1')
n_components: 6
rule_variant: data_driven_emp_cv
alpha / empirical quantile: 0.05 / 0.95


,preprocessing,n_components,rule_variant,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,orientation_score,cv_target_rejection_rate,cv_rule_limit
2,absorbance_sg_d1,6,data_driven_emp_cv,0.75,53.0,0.0,10.0,45.0,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,-0.11798,0.050092,158.208911


## 7. Fit selected model and project validation/test

The model is still trained only on the two calibration batches. The validation batch is used for decision refinement; the test batch is only evaluated after the decision parameters are frozen.

In [13]:
def fit_project_selected(projection_filters, projection_name):
    cv_df, cv_thresholds = calibrate_simca_thresholds_cv(
        object_db=object_db,
        train_filters=TRAIN_FILTERS_CALIBRATION,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
        n_components=SELECTED_N_COMPONENTS,
        alpha=SELECTED_ALPHA,
        m=FIXED_M,
        random_state=RANDOM_STATE,
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
        group_col="object_id",
        n_splits=CV_N_SPLITS,
    )
    cv_summary = summarize_cv_calibration(cv_df, cv_thresholds)

    final_bundle = fit_final_simca_model(
        object_db=object_db,
        train_filters=TRAIN_FILTERS_CALIBRATION,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
        n_components=SELECTED_N_COMPONENTS,
        alpha=SELECTED_ALPHA,
        m=FIXED_M,
        random_state=RANDOM_STATE,
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
    )

    pixel_wide_df, simca_values, X_pixel = project_pixels_with_rule_variants(
        object_db=object_db,
        final_bundle=final_bundle,
        projection_filters=projection_filters,
        cv_thresholds=cv_thresholds,
        rule_variants=[SELECTED_RULE_VARIANT],
    )

    pixel_wide_df = add_pixel_truth_labels(
        pixel_df=pixel_wide_df,
        image_db=image_db,
        object_db=object_db,
        dilation_radius=FIXED_POSITION_DILATION_RADIUS,
    )

    pixel_df = pixel_wide_df.copy()
    rule = SELECTED_RULE_VARIANT
    pixel_df["predicted_peanut_pixel"] = pixel_df[f"pred_{rule}"].astype(bool)
    pixel_df["predicted_label_pixel"] = np.where(pixel_df["predicted_peanut_pixel"], "peanut", "non_peanut")
    pixel_df["rule_statistic"] = pixel_df[f"stat_{rule}"]
    pixel_df["rule_limit"] = pixel_df[f"limit_{rule}"]
    pixel_df["rule_variant"] = rule
    pixel_df["projection_name"] = projection_name

    return {
        "projection_name": projection_name,
        "cv_df": cv_df,
        "cv_thresholds": cv_thresholds,
        "cv_summary": cv_summary,
        "final_bundle": final_bundle,
        "pixel_df": pixel_df,
        "simca_values": simca_values,
        "X_pixel": X_pixel,
    }

validation_selected = fit_project_selected(VALIDATION_FILTERS, "validation")
test_selected = fit_project_selected(TEST_FILTERS, "test")

print("CV calibration summary for the selected configuration:")
display(validation_selected["cv_summary"])

CV calibration summary for the selected configuration:


,rule_variant,stat_col,limit,n,n_rejected_target_cv,rejection_rate_target_cv,acceptance_rate_target_cv,expected_rejection_rate,expected_acceptance_rate,abs_rejection_error
0,simple_emp_cv,simple_chi2_stat,2.082295,3813,191,0.050092,0.949908,0.05,0.95,0.000092
1,alternative_chi2_emp_cv,alternative_chi2_stat,2.813204,3813,191,0.050092,0.949908,0.05,0.95,0.000092
2,data_driven_emp_cv,data_driven_stat,158.208911,3813,191,0.050092,0.949908,0.05,0.95,0.000092
3,alternative_empHQ_emp_cv,alternative_empHQ_stat,1.783779,3813,191,0.050092,0.949908,0.05,0.95,0.000092
4,alternative_chi2_fixed2,alternative_chi2_stat,2.000000,3813,433,0.113559,0.886441,0.05,0.95,0.063559
5,simple_chi2,simple_chi2_stat,1.000000,3813,897,0.235248,0.764752,0.05,0.95,0.185248


## 8. Refine object decision on validation only

This is where we tune the object-level decision threshold and optional border exclusion. This step is not a spectral SIMCA hyperparameter; it is a decision aggregation parameter.

In [14]:
BORDER_WIDTHS = [0, 1, 2, 3]
OBJECT_THRESHOLDS_FOR_DECISION = np.round(np.arange(0.60, 0.96, 0.05), 2)
MIN_CORE_PIXELS = 20

validation_decision_df, validation_decision_tables = border_width_object_threshold_grid(
    pixel_df=validation_selected["pixel_df"],
    object_db=object_db,
    border_widths=BORDER_WIDTHS,
    object_thresholds=OBJECT_THRESHOLDS_FOR_DECISION,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

validation_decision_ranked = rank_simca_results(validation_decision_df)

cols_decision = [
    "border_width", "object_threshold", "tp", "fn", "fp", "tn",
    "fn_rate", "fp_rate", "peanut_sensitivity", "almond_specificity",
    "accuracy", "f1_score", "balanced_accuracy", "mean_core_fraction", "fallback_object_rate", "orientation_score",
]
display(validation_decision_ranked[cols_decision].head(30))

,border_width,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,mean_core_fraction,fallback_object_rate,orientation_score
0,0,0.75,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,1.000000,0.009259,-0.117980
1,1,0.80,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,0.688626,0.083333,-0.117980
2,2,0.75,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,0.391751,0.388889,-0.117980
3,2,0.80,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,0.391751,0.388889,-0.117980
4,1,0.75,53,0,11,44,0.000000,0.200000,1.000000,0.800000,0.898148,0.905983,0.900000,0.688626,0.083333,-0.136738
5,3,0.75,53,0,11,44,0.000000,0.200000,1.000000,0.800000,0.898148,0.905983,0.900000,0.117780,0.861111,-0.136738
6,0,0.80,52,1,10,45,0.018868,0.181818,0.981132,0.818182,0.898148,0.904348,0.899657,1.000000,0.009259,-0.307317
7,3,0.80,52,1,10,45,0.018868,0.181818,0.981132,0.818182,0.898148,0.904348,0.899657,0.117780,0.861111,-0.307317
8,1,0.85,51,2,9,46,0.037736,0.163636,0.962264,0.836364,0.898148,0.902655,0.899314,0.688626,0.083333,-0.477899
9,2,0.85,50,3,8,47,0.056604,0.145455,0.943396,0.854545,0.898148,0.900901,0.898971,0.391751,0.388889,-0.648484


In [15]:
SELECTED_DECISION_ROW = validation_decision_ranked.iloc[0].copy()
SELECTED_BORDER_WIDTH = int(SELECTED_DECISION_ROW["border_width"])
SELECTED_OBJECT_THRESHOLD = float(SELECTED_DECISION_ROW["object_threshold"])

print("Selected object-decision parameters from validation:")
print("border_width:", SELECTED_BORDER_WIDTH)
print("object_threshold:", SELECTED_OBJECT_THRESHOLD)
display(pd.DataFrame([SELECTED_DECISION_ROW])[cols_decision])

Selected object-decision parameters from validation:
border_width: 0
object_threshold: 0.75


,border_width,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,mean_core_fraction,fallback_object_rate,orientation_score
0,0.0,0.75,53.0,0.0,10.0,45.0,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,1.0,0.009259,-0.11798


## 9. Final test evaluation

The following cell applies the selected spectral configuration and selected object-decision parameters to the test batch. Do not modify the chosen parameters based on this output.

In [16]:
validation_object_df = aggregate_pixel_predictions_to_objects_core(
    pixel_df=validation_selected["pixel_df"],
    object_db=object_db,
    object_threshold=SELECTED_OBJECT_THRESHOLD,
    border_width=SELECTED_BORDER_WIDTH,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

test_object_df = aggregate_pixel_predictions_to_objects_core(
    pixel_df=test_selected["pixel_df"],
    object_db=object_db,
    object_threshold=SELECTED_OBJECT_THRESHOLD,
    border_width=SELECTED_BORDER_WIDTH,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

validation_metrics = binary_detection_metrics(
    validation_object_df,
    true_col="true_peanut_object",
    pred_col="predicted_peanut_object",
)
validation_metrics["set"] = "validation"

test_metrics = binary_detection_metrics(
    test_object_df,
    true_col="true_peanut_object",
    pred_col="predicted_peanut_object",
)
test_metrics["set"] = "test"

final_comparison_df = add_orientation_columns(pd.DataFrame([validation_metrics, test_metrics]))
final_comparison_df["preprocessing"] = SELECTED_PREPROCESSING
final_comparison_df["n_components"] = SELECTED_N_COMPONENTS
final_comparison_df["rule_variant"] = SELECTED_RULE_VARIANT
final_comparison_df["border_width"] = SELECTED_BORDER_WIDTH
final_comparison_df["object_threshold"] = SELECTED_OBJECT_THRESHOLD

display(final_comparison_df[[
    "set", "tp", "fn", "fp", "tn", "fn_rate", "fp_rate",
    "peanut_sensitivity", "almond_specificity", "accuracy", "f1_score", "balanced_accuracy",
    "preprocessing", "n_components", "rule_variant", "border_width", "object_threshold",
]])

,set,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,preprocessing,n_components,rule_variant,border_width,object_threshold
0,validation,53,0,10,45,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,absorbance_sg_d1,6,data_driven_emp_cv,0,0.75
1,test,29,0,15,33,0.0,0.312500,1.0,0.687500,0.805195,0.794521,0.843750,absorbance_sg_d1,6,data_driven_emp_cv,0,0.75


## 10. Error diagnostics

Use these plots to understand whether FP/FN are mostly on borders or in the core of the object.

In [17]:
print("Validation pixel errors by border zone")
display(summarize_pixel_errors_by_border_zone(
    pixel_df=validation_selected["pixel_df"],
    object_db=object_db,
    border_width=SELECTED_BORDER_WIDTH,
))

print("Test pixel errors by border zone")
display(summarize_pixel_errors_by_border_zone(
    pixel_df=test_selected["pixel_df"],
    object_db=object_db,
    border_width=SELECTED_BORDER_WIDTH,
))

Validation pixel errors by border zone


,zone,border_width,n_pixels,tp,tn,fp,fn,fp_rate,fn_rate,pixel_accuracy
0,core,0,6812,3000,1982,1633,197,0.451729,0.06162,0.731356


Test pixel errors by border zone


,zone,border_width,n_pixels,tp,tn,fp,fn,fp_rate,fn_rate,pixel_accuracy
0,core,0,8401,3001,2626,2611,163,0.498568,0.051517,0.669801


In [18]:
def add_object_error_type(object_df):
    df = object_df.copy()
    df["object_error_type"] = "NA"
    mask = df["true_peanut_object"].notna() & df["predicted_peanut_object"].notna()
    y_true = df.loc[mask, "true_peanut_object"].astype(bool)
    y_pred = df.loc[mask, "predicted_peanut_object"].astype(bool)
    df.loc[mask & y_true & y_pred, "object_error_type"] = "TP"
    df.loc[mask & (~y_true) & (~y_pred), "object_error_type"] = "TN"
    df.loc[mask & (~y_true) & y_pred, "object_error_type"] = "FP"
    df.loc[mask & y_true & (~y_pred), "object_error_type"] = "FN"
    return df

validation_object_df = add_object_error_type(validation_object_df)
test_object_df = add_object_error_type(test_object_df)

print("Validation object errors")
display(validation_object_df.groupby(["source_image", "object_error_type"], as_index=False).size())
print("Test object errors")
display(test_object_df.groupby(["source_image", "object_error_type"], as_index=False).size())

Validation object errors


,source_image,object_error_type,size
0,almond3,FP,10
1,almond3,TN,45
2,peanut3,TP,53


Test object errors


,source_image,object_error_type,size
0,almond4,FP,15
1,almond4,TN,33
2,peanut4,TP,29


In [19]:
# Visualize the most problematic test images.
error_images = (
    test_object_df[test_object_df["object_error_type"].isin(["FP", "FN"])]
    ["source_image"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

for image_key in error_images[:6]:
    plot_pixel_error_overlay(
        image_key=image_key,
        image_db=image_db,
        pixel_df=test_selected["pixel_df"],
        title=f"Test pixel error map — {image_key}",
    )

## 11. Optional deployment projection on mixtures

This section is separate from model selection and test evaluation. Once the orientation is reported, you may refit on all pure peanut batches before projecting mixtures. Keep the selected preprocessing, number of components, rule and object decision parameters fixed.

In [20]:
TRAIN_FILTERS_DEPLOYMENT = {
    "sample_kind": ["pure"],
    "object_nut_type": ["peanut"],
    "batch": [1, 2, 3, 4],
}

# Recalibrate empirical 95% thresholds on all available pure peanut batches.
deployment_cv_df, deployment_cv_thresholds = calibrate_simca_thresholds_cv(
    object_db=object_db,
    train_filters=TRAIN_FILTERS_DEPLOYMENT,
    matrix_method=FIXED_MATRIX_METHOD,
    preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
    n_components=SELECTED_N_COMPONENTS,
    alpha=SELECTED_ALPHA,
    m=FIXED_M,
    random_state=RANDOM_STATE,
    replace=False,
    wavelengths=wavelengths,
    sg_window_length=FIXED_SG_WINDOW_LENGTH,
    sg_polyorder=FIXED_SG_POLYORDER,
    group_col="object_id",
    n_splits=CV_N_SPLITS,
)

deployment_bundle = fit_final_simca_model(
    object_db=object_db,
    train_filters=TRAIN_FILTERS_DEPLOYMENT,
    matrix_method=FIXED_MATRIX_METHOD,
    preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
    n_components=SELECTED_N_COMPONENTS,
    alpha=SELECTED_ALPHA,
    m=FIXED_M,
    random_state=RANDOM_STATE,
    replace=False,
    wavelengths=wavelengths,
    sg_window_length=FIXED_SG_WINDOW_LENGTH,
    sg_polyorder=FIXED_SG_POLYORDER,
)

mixture_pixel_wide_df, _, _ = project_pixels_with_rule_variants(
    object_db=object_db,
    final_bundle=deployment_bundle,
    projection_filters=PROJECTION_FILTERS_MIXTURES,
    cv_thresholds=deployment_cv_thresholds,
    rule_variants=[SELECTED_RULE_VARIANT],
)

mixture_pixel_wide_df = add_pixel_truth_labels(
    pixel_df=mixture_pixel_wide_df,
    image_db=image_db,
    object_db=object_db,
    dilation_radius=FIXED_POSITION_DILATION_RADIUS,
)

mixture_pixel_df = mixture_pixel_wide_df.copy()
mixture_pixel_df["predicted_peanut_pixel"] = mixture_pixel_df[f"pred_{SELECTED_RULE_VARIANT}"].astype(bool)
mixture_pixel_df["rule_statistic"] = mixture_pixel_df[f"stat_{SELECTED_RULE_VARIANT}"]
mixture_pixel_df["rule_limit"] = mixture_pixel_df[f"limit_{SELECTED_RULE_VARIANT}"]

mixture_object_df = aggregate_pixel_predictions_to_objects_core(
    pixel_df=mixture_pixel_df,
    object_db=object_db,
    object_threshold=SELECTED_OBJECT_THRESHOLD,
    border_width=SELECTED_BORDER_WIDTH,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

display(mixture_object_df.head())

c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


,object_id,source_image,n_pixels_total,n_pixels_core,n_pixels_border,n_pixels_decision,decision_used_core,border_width,min_core_pixels,n_predicted_peanut_pixels,peanut_pixel_ratio,predicted_peanut_object,predicted_label_object,object_threshold,true_peanut_pixel_ratio_decision,true_peanut_pixel_ratio_total,true_peanut_object,true_label_object,area_pixels,batch,sample_kind,object_nut_type,centroid_row,centroid_col
0,alm1pea1_obj001,alm1pea1,84,84,0,84,True,0,20,27,0.321429,False,non_peanut,0.75,0.0,0.0,False,non_peanut,84,None,mixture,unknown,85.726190,45.107143
1,alm1pea1_obj002,alm1pea1,73,73,0,73,True,0,20,28,0.383562,False,non_peanut,0.75,0.0,0.0,False,non_peanut,73,None,mixture,unknown,87.739726,123.767123
2,alm1pea1_obj003,alm1pea1,91,91,0,91,True,0,20,90,0.989011,True,peanut,0.75,1.0,1.0,True,peanut,91,None,mixture,unknown,90.032967,157.340659
3,alm1pea1_obj004,alm1pea1,73,73,0,73,True,0,20,69,0.945205,True,peanut,0.75,1.0,1.0,True,peanut,73,None,mixture,unknown,93.589041,90.164384
4,alm1pea1_obj005,alm1pea1,126,126,0,126,True,0,20,88,0.698413,False,non_peanut,0.75,0.0,0.0,False,non_peanut,126,None,mixture,unknown,98.825397,70.579365


In [21]:
mixture_object_df[(mixture_object_df['predicted_peanut_object']==False) & (mixture_object_df['true_peanut_object']==True)]

,object_id,source_image,n_pixels_total,n_pixels_core,n_pixels_border,n_pixels_decision,decision_used_core,border_width,min_core_pixels,n_predicted_peanut_pixels,peanut_pixel_ratio,predicted_peanut_object,predicted_label_object,object_threshold,true_peanut_pixel_ratio_decision,true_peanut_pixel_ratio_total,true_peanut_object,true_label_object,area_pixels,batch,sample_kind,object_nut_type,centroid_row,centroid_col
28,alm1pea1_obj029,alm1pea1,60,60,0,60,True,0,20,42,0.700000,False,non_peanut,0.75,1.0,1.0,True,peanut,60,None,mixture,unknown,232.000000,199.700000
309,alm3pea1_obj004,alm3pea1,102,102,0,102,True,0,20,72,0.705882,False,non_peanut,0.75,1.0,1.0,True,peanut,102,None,mixture,unknown,101.794118,166.019608
352,alm3pea2_obj005,alm3pea2,63,63,0,63,True,0,20,37,0.587302,False,non_peanut,0.75,1.0,1.0,True,peanut,63,None,mixture,unknown,111.825397,139.365079
410,alm3pea3_obj024,alm3pea3,54,54,0,54,True,0,20,34,0.629630,False,non_peanut,0.75,1.0,1.0,True,peanut,54,None,mixture,unknown,223.462963,221.259259


5 FN

In [22]:
mixture_object_df[(mixture_object_df['predicted_peanut_object']==True) & (mixture_object_df['true_peanut_object']==False)]

,object_id,source_image,n_pixels_total,n_pixels_core,n_pixels_border,n_pixels_decision,decision_used_core,border_width,min_core_pixels,n_predicted_peanut_pixels,peanut_pixel_ratio,predicted_peanut_object,predicted_label_object,object_threshold,true_peanut_pixel_ratio_decision,true_peanut_pixel_ratio_total,true_peanut_object,true_label_object,area_pixels,batch,sample_kind,object_nut_type,centroid_row,centroid_col
134,alm1pea4_obj013,alm1pea4,118,118,0,118,True,0,20,107,0.906780,True,peanut,0.75,0.0,0.0,False,non_peanut,118,None,mixture,unknown,176.898305,109.906780
311,alm3pea1_obj006,alm3pea1,69,69,0,69,True,0,20,53,0.768116,True,peanut,0.75,0.0,0.0,False,non_peanut,69,None,mixture,unknown,106.637681,97.057971
312,alm3pea1_obj007,alm3pea1,99,99,0,99,True,0,20,86,0.868687,True,peanut,0.75,0.0,0.0,False,non_peanut,99,None,mixture,unknown,112.111111,131.282828
313,alm3pea1_obj008,alm3pea1,89,89,0,89,True,0,20,80,0.898876,True,peanut,0.75,0.0,0.0,False,non_peanut,89,None,mixture,unknown,122.573034,65.134831
316,alm3pea1_obj011,alm3pea1,72,72,0,72,True,0,20,64,0.888889,True,peanut,0.75,0.0,0.0,False,non_peanut,72,None,mixture,unknown,136.236111,182.583333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578,alm4pea4_obj021,alm4pea4,126,126,0,126,True,0,20,95,0.753968,True,peanut,0.75,0.0,0.0,False,non_peanut,126,None,mixture,unknown,211.984127,148.230159
579,alm4pea4_obj022,alm4pea4,111,111,0,111,True,0,20,89,0.801802,True,peanut,0.75,0.0,0.0,False,non_peanut,111,None,mixture,unknown,226.162162,69.207207
580,alm4pea4_obj023,alm4pea4,141,141,0,141,True,0,20,118,0.836879,True,peanut,0.75,0.0,0.0,False,non_peanut,141,None,mixture,unknown,231.241135,198.241135
592,alm4pea4_obj035,alm4pea4,153,153,0,153,True,0,20,142,0.928105,True,peanut,0.75,0.0,0.0,False,non_peanut,153,None,mixture,unknown,308.137255,84.679739


57 FP

In [49]:
mixture_object_df[(mixture_object_df['predicted_peanut_object']==True) & (mixture_object_df['true_peanut_object']==False)]['source_image'].value_counts()

source_image
alm4pea3    11
alm4pea2     9
alm4pea4     8
alm3pea1     6
alm4pea1     6
alm3pea2     5
alm3pea4     5
alm3pea3     4
alm1pea4     3
Name: count, dtype: int64

# Sum up

By best accuracy:
- data_driven_emp_cv/ simple_emp_cv
- absorbance_sg_d1
- 6, 7, 11 compo

test with data_driven_emp_cv, absorbance_sg_d1, 7 compo, 85 threshold, 1 px border removed => 5 FN, 57 FP

test with data_driven_emp_cv, absorbance_sg_d1, 6 compo, 75 threshold, 0 px border removed => 4 FN, 66 FP
